In [1]:
from mpramnist.Fromel2025.dataset import FromelDataset
from mpramnist.Fromel2025.trainer import MaskedMSE, MaskedPearsonCorrCoef
from mpramnist.Fromel2025.trainer import LitModel_Fromel

from mpramnist.models import HumanLegNet
from mpramnist.models import initialize_weights

import mpramnist.transforms as t
import mpramnist.target_transforms as t_t

import torch
import torch.nn as nn
import torch.utils.data as data

import lightning.pytorch as L

In [2]:
FromelDataset.HSPC_TARGETS

['State_1M',
 'State_2D',
 'State_3E',
 'State_4M',
 'State_5M',
 'State_6N',
 'State_7M']

In [3]:
# preprocessing
train_transform = t.Compose(
    [
        t.AddFlanks(FromelDataset.CONSTANT_LEFT_FLANK, FromelDataset.CONSTANT_RIGHT_FLANK),
        t.AddFlanks("", FromelDataset.RIGHT_FLANK),  # this is original parameters for human_legnet
        t.RightCrop(245, 270),  # this is using for shifting
        t.LeftCrop(245, 245),
        t.ReverseComplement(0.5),
        t.AddFeatureChannels(['batch']),
        t.Seq2Tensor(),
    ]
)
test_transform = t.Compose(
    [
        t.AddFlanks(FromelDataset.CONSTANT_LEFT_FLANK, FromelDataset.CONSTANT_RIGHT_FLANK),
        t.LeftCrop(245, 245),
        t.ReverseComplement(0),
        t.AddFeatureChannels(['batch']),
        t.Seq2Tensor(),
    ]
)

In [4]:
train_dataset = FromelDataset(split="train", transform=train_transform, targets=FromelDataset.HSPC_TARGETS, root="../data/")
val_dataset = FromelDataset(split="val", transform=test_transform, targets=FromelDataset.HSPC_TARGETS, root="../data/")

In [6]:
val_dataset[0]

(tensor([[0., 0., 0.,  ..., 0., 0., 1.],
         [1., 0., 0.,  ..., 1., 0., 0.],
         [0., 1., 1.,  ..., 0., 1., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]]),
 array([ 0.4612903 , -0.07443686, -0.09789075, -0.16048531,  0.36409938,
         0.7970658 ,  0.20762111], dtype=float32))

In [5]:
in_channels = len(train_dataset[0][0])
out_channels = len(train_dataset[0][1])

In [ ]:
train_dl  = data.DataLoader(train_dataset, batch_size=1024, num_workers=16, shuffle=True)
val_dl  = data.DataLoader(val_dataset, batch_size=4096*2, num_workers=16, shuffle=False)

In [7]:
model = HumanLegNet(
                in_ch=in_channels,
                output_dim=out_channels,
                stem_ch=64,
                stem_ks=11,
                ef_ks=9,
                ef_block_sizes=[80, 96, 112, 128],
                pool_sizes=[2, 2, 2, 2],
                resize_factor=4,
        )
model.apply(initialize_weights)
        
seq_model = LitModel_Fromel(
    model=model, 
    activity_columns=FromelDataset.HSPC_TARGETS,
    loss=MaskedMSE(),
        weight_decay=2e-1,
        lr=0.005, 
    print_each=20,
)

In [8]:
trainer = L.Trainer(
            accelerator="gpu",
            devices=[0],
            max_epochs=1,
            gradient_clip_val=1,
            precision="16-mixed",
            enable_progress_bar=True,
            num_sanity_val_steps=0,
        )
trainer.fit(seq_model, 
            train_dataloaders=train_dl,
            val_dataloaders=val_dl)

Using 16bit Automatic Mixed Precision (AMP)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/nios/miniconda3/envs/mpra/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly

Epoch 0: 100%|██████████| 45/45 [00:12<00:00,  3.71it/s, v_num=7, train_loss_step=0.109, val_loss=0.116, val_State_1M_pearson=0.182, val_State_2D_pearson=0.189, val_State_3E_pearson=0.179, val_State_4M_pearson=0.241, val_State_5M_pearson=0.163, val_State_6N_pearson=0.154, val_State_7M_pearson=0.135, val_pearson=0.178, train_loss_epoch=0.107]

`Trainer.fit` stopped: `max_epochs=1` reached.


Epoch 0: 100%|██████████| 45/45 [00:12<00:00,  3.67it/s, v_num=7, train_loss_step=0.109, val_loss=0.116, val_State_1M_pearson=0.182, val_State_2D_pearson=0.189, val_State_3E_pearson=0.179, val_State_4M_pearson=0.241, val_State_5M_pearson=0.163, val_State_6N_pearson=0.154, val_State_7M_pearson=0.135, val_pearson=0.178, train_loss_epoch=0.107]


In [9]:
test_dataset = FromelDataset(split='test', transform=test_transform, targets=FromelDataset.HSPC_TARGETS, root = "../data")
test_dl  = data.DataLoader(test_dataset, batch_size=4096*2, num_workers=64, shuffle=False)

In [10]:
trainer.test(seq_model, dataloaders=test_dl)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Testing DataLoader 0: 100%|██████████| 1/1 [00:00<00:00, 10.03it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  test_State_1M_pearson     0.22124142944812775
  test_State_2D_pearson     0.2122603952884674
  test_State_3E_pearson     0.1972126066684723
  test_State_4M_pearson     0.2941748797893524
  test_State_5M_pearson     0.18573808670043945
  test_State_6N_pearson     0.18276771903038025
  test_State_7M_pearson     0.17010177671909332
        test_loss           0.1251792162656784
      test_pearson          0.20907099545001984
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'test_loss': 0.1251792162656784,
  'test_State_1M_pearson': 0.22124142944812775,
  'test_State_2D_pearson': 0.2122603952884674,
  'test_State_3E_pearson': 0.1972126066684723,
  'test_State_4M_pearson': 0.2941748797893524,
  'test_State_5M_pearson': 0.18573808670043945,
  'test_State_6N_pearson': 0.18276771903038025,
  'test_State_7M_pearson': 0.17010177671909332,
  'test_pearson': 0.20907099545001984}]

In [11]:
test_dataset = FromelDataset(split='genome', transform=test_transform, targets=FromelDataset.HSPC_TARGETS, root = "../data")
test_dl  = data.DataLoader(test_dataset, batch_size=4096*2, num_workers=64, shuffle=False)
trainer.test(seq_model, dataloaders=test_dl)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Testing DataLoader 0: 100%|██████████| 1/1 [00:00<00:00, 18.01it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  test_State_1M_pearson     0.03455352783203125
  test_State_2D_pearson    -0.02404255047440529
  test_State_3E_pearson    -0.09706783294677734
  test_State_4M_pearson    -0.09381520003080368
  test_State_5M_pearson             nan
  test_State_6N_pearson     0.0630899965763092
  test_State_7M_pearson    -0.09785396605730057
        test_loss           0.11701618880033493
      test_pearson         -0.03585600480437279
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'test_loss': 0.11701618880033493,
  'test_State_1M_pearson': 0.03455352783203125,
  'test_State_2D_pearson': -0.02404255047440529,
  'test_State_3E_pearson': -0.09706783294677734,
  'test_State_4M_pearson': -0.09381520003080368,
  'test_State_5M_pearson': nan,
  'test_State_6N_pearson': 0.0630899965763092,
  'test_State_7M_pearson': -0.09785396605730057,
  'test_pearson': -0.03585600480437279}]